# RAPID Survival (Cox) — Tutorial Notebook

This notebook walks you through using the RAPID **Survival Cox** pipeline from end to end.

By the end of this tutorial, you will know how to:

* Instantiate a survival model using the **RAPID Pipeline Factory**.
* Fit the model using duration and binary event variables.
* Interpret **assumption tests**, **performance metrics**, and **domain-specific plots** like Kaplan-Meier curves.
* Use custom formulas to test interactions between variables.

This pipeline is designed for **epidemiological and clinical research**, focusing on "time-to-event" outcomes (e.g., time to discharge, time to death, or time to recovery).

---

## Setup and Installation
Before starting, ensure the RAPID
package is installed in your environment.

```bash
pip install "isaric3.0-rapid/dist/isaric-0.1.0-py3-none-any.whl
```

Then import the factory and any other dependencies you need.

In [1]:
!pip install gdown

import gdown

#url = "https://drive.google.com/uc?export=download&id=1OpP-r3YzKGTKYHijHE-yAgFNYLG-5vj-"
url = "https://drive.google.com/uc?export=download&id=1wDgkpjlRc_K2263RL_lq_cPxiDm8EHv7"

gdown.download(url, 'isaric-0.1.0-py3-none-any.whl', quiet=False)

!pip install isaric-0.1.0-py3-none-any.whl

Downloading...
From: https://drive.google.com/uc?export=download&id=1wDgkpjlRc_K2263RL_lq_cPxiDm8EHv7
To: /mnt/sandbox-SSD-2/sarah/isaric3.0-rapid/jupyter/isaric-0.1.0-py3-none-any.whl
100%|██████████| 75.0k/75.0k [00:00<00:00, 860kB/s]


Processing ./isaric-0.1.0-py3-none-any.whl
isaric is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.


In [2]:
import pandas as pd
import warnings
from isaric.pipelines.pipeline_factory import RAPID_PipelineFactory
# Ignore standard runtime warnings during optimization steps
warnings.filterwarnings('ignore', category=RuntimeWarning)

## The Pipeline Factory

All RAPID pipelines are created through the **`RAPID_PipelineFactory`**. This provides a single, consistent entry point for creating any supported pipeline by name, ensuring your code remains clean and standardized.

In [3]:

# Instantiate the factory
factory = RAPID_PipelineFactory()

# See all pipelines available out of the box
print(f"Available pipelines: {factory.available()}")


Available pipelines: ['glm', 'logistic', 'survival']




## Preparing Your Data

Survival models require two mandatory outcome columns:

* **Duration**: A continuous variable representing the follow-up time.
* **Event**: A binary indicator (1 if the event occurred, 0 for censoring/not occurred).

Below is an example using a simulated clinical dataset.

In [6]:
# choose your data file
#from google.colab import files
# Upload file
#uploaded = files.upload()
#df_model = pd.read_csv('df_model.csv')
from pathlib import Path
data_path_df_model = Path.cwd() .parent/ 'data' / 'df_model.csv'
df_model = pd.read_csv(data_path_df_model)

In [7]:
import numpy as np

df_model.head()

,period,Gender,SofaScore,Saps3Points,Beds,Age,Idade_Agrupada,Idade_Agrupada2,MFI_Agregado,AdmissionSourceName,...,HospitalDischargeCode_trunc,HospitalDischargeCode_trunc_bin,AnoAgrupado_HospitalDischarge_trunc,UnitDischargeCode_trunc,UnitDischargeCode,HospitalLengthStay,HospitalLengthStay_trunc,UnitLengthStay,InternacaoPrevia,HospitalDischargeCode
0,2018-2019,M,NaN,59,10,63,60-69,<65,PreFrail&Frail (MFI>=1,Emergency room,...,A,0,2018-2019_A,A,A,23,23,5,0,A
1,2018-2019,F,0.0,42,22,21,<40,<65,Non-frail (MFI=0),Emergency room,...,A,0,2018-2019_A,A,A,7,7,5,0,A
2,2018-2019,F,1.0,54,11,71,70-79,65-79,PreFrail&Frail (MFI>=1,Emergency room,...,A,0,2018-2019_A,A,A,6,6,4,0,A
3,2022-2023,M,2.0,46,32,45,40-49,<65,PreFrail&Frail (MFI>=1,Emergency room,...,A,0,2022-2023_A,A,A,8,8,4,0,A
4,2018-2019,F,6.0,59,31,69,60-69,65-79,PreFrail&Frail (MFI>=1,Emergency room,...,A,0,2018-2019_A,A,A,36,36,5,0,A



## Creating and Fitting the Model

Pass your DataFrame, the time variable (`duration_var`), the event variable (`dependent_var`), and your predictors (`independent_vars`) to `factory.create()`.



In [8]:
model = factory.create(
    "survival",
    data=df_model,
    duration_var="HospitalLengthStay_trunc",
    dependent_var="HospitalDischargeCode_trunc_bin",
    independent_vars=['period', 'Idade_Agrupada2', 'ChronicHealthStatusName', 'obesity',
    'IsImmunossupression', 'IsSteroidsUse', 'IsSevereCopd', 'IsChfNyha',
    'cancer']
)

### Fitting with Labels and Penalization

The `.fit()` method estimates the Hazard Ratios. You can pass a `labels` dictionary for clean reporting and a `penalizer` for L2 regularization to handle multicollinearity or small sample sizes.



In [9]:
model.fit(
    labels={'obesity': 'Clinical Obesity', 'cancer': 'Malignancy'
    },
    penalizer=0.1,
    cross_val=True,
    n_splits=5
)


TypeError: CoxPHFitter.fit() got an unexpected keyword argument 'duration_var'. Did you mean 'duration_col'?

During `fit()`, the pipeline automatically:
- Fits the Survival-Cox
- Computes all performance metrics
- Runs k-fold cross-validation (if `cross_val=True`)

---

## Results Table (Hazard Ratios)

In survival analysis, we interpret the **Hazard Ratio (HR)** rather than linear coefficients.



In [ ]:

model.summary_df

**Outputs**
<dd>

The output consists of a pandas DataFrame with the following columns for each predictor:

<ul>
  <li><b>HazardRatio</b>: Hazard Ratio.</li>
  <li><b>LowerCI</b>: Lower bound of the 95% confidence interval.</li>
  <li><b>UpperCI</b>: Upper bound of the 95% confidence interval.</li>
  <li><b>p-value</b>: P-value for statistical significance. </li>
</ul>

<dt>

### How to interpret:

* **Hazard Ratio (HR) > 1**: Indicates an increased risk of the event (e.g., higher probability of death).
* **Hazard Ratio (HR) < 1**: Indicates a protective factor (lower risk of the event).
* **Hazard Ratio (HR) = 1**: Indicatesthat the predictor has no effect on survival.
* **95% Confidence Interval**: If the interval **does not include 1.0**, the association is statistically significant at $p < 0.05$.

---

## Assumption Tests

The core assumption of the Cox model is **Proportional Hazards**.



In [ ]:

model.summary(performance=True, assumptions=True, plots=["forest_plot"])



---
## Performance and Cross-Validation
Unlike GLM which uses R², survival models use the C-index (Concordance Index).



In [ ]:
model.summary(performance=True)


### Concordance Index (C-index)

Measures the discriminative power of the model.

* **0.5**: No better than random chance.
* **0.7**: Good discrimination.
* **1.0**: Perfect discrimination.

---

- C-index: Measures how well the model predicts the order of events. A value of 0.7 or higher generally indicates a good model.

- Mean CV C-index: The average performance across cross-validation folds. If the CV score is significantly lower than the training score, the model may be overfitting.

## Diagnostic and Visualization Plots

Survival analysis requires specific plots to understand the dynamics of the event.

### Forest Plot

Displays Hazard Ratios and their 95% CIs. It is ideal for comparing the relative impact of different predictors at a glance.



In [ ]:
model.summary(plots=["forest_plot"])




### Kaplan-Meier / Survival Curve

Shows the probability of "surviving" (not experiencing the event) over time.



In [ ]:

model.summary(plots=["survival_curve"])



In [ ]:
model.summary(plots= ["roc_auc", "brier_score"], target_time=14.0)

In [ ]:
model.summary(performance=True)

---

## Summary Table

| Step | Action |
| --- | --- |
| **Factory** | Created via `RAPID_PipelineFactory`. |
| **Data Prep** | Defined `duration_var` (time) and `dependent_var` (event). |
| **Assumptions** | Tested Proportional Hazards via Schoenfeld residuals. |
| **Performance** | Evaluated via C-Index and Log-Likelihood. |
| **Plots** | Forest plots, Kaplan-Meier curves, and Martingale residuals. |
